# CareerLens AI
## Intelligent Resume–Job Matching and Skill Gap Analysis System Using NLP and Machine Learning

This notebook contains the complete practical pipeline for the MCA project.

### Practical Workflow

1. Environment setup
2. Project folder creation
3. Resume dataset acquisition
4. Job dataset acquisition
5. Data inspection and cleaning
6. Exploratory data analysis
7. NLP text preparation
8. Logistic Regression model
9. Linear SVM model
10. Random Forest model
11. Model comparison
12. Final model training
13. Skill vocabulary creation
14. Job matching engine
15. Matching evaluation
16. Model and data export
17. SQLite database creation
18. Streamlit application generation
19. Deployment file generation
20. Final verification

Run the notebook from top to bottom in VS Code before starting the Streamlit application.


## 1. Environment Setup

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib plotly wordcloud joblib PyMuPDF python-docx streamlit requests

## 2. Imports and Project Structure

In [ ]:
from pathlib import Path
import json
import re
import sqlite3
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import requests
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
STREAMLIT_DIR = ROOT / ".streamlit"

DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
STREAMLIT_DIR.mkdir(exist_ok=True)

print(ROOT)

## 3. Data Sources

The project uses a public labelled resume dataset and a public jobs dataset.

The notebook also includes a fallback job catalogue so that the complete practical work remains executable if the remote jobs source is temporarily unavailable.


In [ ]:
RESUME_URL = "https://raw.githubusercontent.com/611noorsaeed/Resume-Screening-App/main/UpdatedResumeDataSet.csv"
JOB_URL = "https://raw.githubusercontent.com/Fatmaayadi/Job-Roles-Prediction/main/data/csv/jobs.csv"

resume_path = DATA_DIR / "UpdatedResumeDataSet.csv"
job_path = DATA_DIR / "jobs_source.csv"

if not resume_path.exists():
    response = requests.get(RESUME_URL, timeout=60)
    response.raise_for_status()
    resume_path.write_bytes(response.content)

resume_df = pd.read_csv(resume_path)
resume_df.head()

## 4. Resume Data Inspection

In [ ]:
print("Shape:", resume_df.shape)
print("Columns:", resume_df.columns.tolist())
print("Missing values:")
display(resume_df.isna().sum().to_frame("Missing"))
print("Duplicate rows:", resume_df.duplicated().sum())
print("Categories:", resume_df["Category"].nunique())
display(resume_df["Category"].value_counts().to_frame("Count"))

## 5. Resume Data Cleaning

In [ ]:
resume_df = resume_df[["Category", "Resume"]].copy()
resume_df["Category"] = resume_df["Category"].astype(str).str.strip()
resume_df["Resume"] = resume_df["Resume"].astype(str).str.strip()
resume_df = resume_df.replace({"": np.nan}).dropna()
resume_df = resume_df.drop_duplicates(subset=["Category", "Resume"]).reset_index(drop=True)
resume_df["resume_length"] = resume_df["Resume"].str.split().str.len()

print("Clean shape:", resume_df.shape)
display(resume_df.head())

## 6. Resume Exploratory Data Analysis

In [ ]:
category_counts = resume_df["Category"].value_counts().reset_index()
category_counts.columns = ["Category", "Count"]

fig = px.bar(
    category_counts.sort_values("Count"),
    x="Count",
    y="Category",
    orientation="h",
    title="Resume Category Distribution",
    template="plotly_dark"
)
fig.update_layout(height=720)
fig.show()

In [ ]:
fig = px.histogram(
    resume_df,
    x="resume_length",
    nbins=35,
    title="Resume Length Distribution",
    template="plotly_dark"
)
fig.update_layout(xaxis_title="Words per Resume", yaxis_title="Number of Resumes")
fig.show()

## 7. NLP Text Preparation

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-z0-9+#.\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

resume_df["clean_resume"] = resume_df["Resume"].apply(clean_text)

display(resume_df[["Category", "clean_resume"]].head())

## 8. Train-Test Split

In [ ]:
X = resume_df["clean_resume"]
y = resume_df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training categories:", y_train.nunique())

## 9. Model 1 — Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=25000
    )),
    ("model", LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)

logistic_accuracy = accuracy_score(y_test, logistic_pred)
logistic_f1 = f1_score(y_test, logistic_pred, average="macro")

print("Accuracy:", round(logistic_accuracy, 4))
print("Macro F1:", round(logistic_f1, 4))
print(classification_report(y_test, logistic_pred, zero_division=0))

In [ ]:
logistic_cm = confusion_matrix(y_test, logistic_pred, labels=logistic_model.classes_)

fig = px.imshow(
    logistic_cm,
    x=logistic_model.classes_,
    y=logistic_model.classes_,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Logistic Regression Confusion Matrix",
    template="plotly_dark",
    aspect="auto"
)
fig.update_layout(height=850)
fig.show()

## 10. Model 2 — Linear Support Vector Machine

In [ ]:
svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=25000
    )),
    ("model", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average="macro")

print("Accuracy:", round(svm_accuracy, 4))
print("Macro F1:", round(svm_f1, 4))
print(classification_report(y_test, svm_pred, zero_division=0))

In [ ]:
svm_cm = confusion_matrix(y_test, svm_pred, labels=svm_model.classes_)

fig = px.imshow(
    svm_cm,
    x=svm_model.classes_,
    y=svm_model.classes_,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Linear SVM Confusion Matrix",
    template="plotly_dark",
    aspect="auto"
)
fig.update_layout(height=850)
fig.show()

## 11. Model 3 — Random Forest

In [ ]:
rf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=12000
)

X_train_rf = rf_vectorizer.fit_transform(X_train)
X_test_rf = rf_vectorizer.transform(X_test)

rf_model = RandomForestClassifier(
    n_estimators=350,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train)
rf_pred = rf_model.predict(X_test_rf)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average="macro")

print("Accuracy:", round(rf_accuracy, 4))
print("Macro F1:", round(rf_f1, 4))
print(classification_report(y_test, rf_pred, zero_division=0))

In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred, labels=rf_model.classes_)

fig = px.imshow(
    rf_cm,
    x=rf_model.classes_,
    y=rf_model.classes_,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Random Forest Confusion Matrix",
    template="plotly_dark",
    aspect="auto"
)
fig.update_layout(height=850)
fig.show()

## 12. Model Comparison

In [ ]:
model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Linear SVM", "Random Forest"],
    "Accuracy": [logistic_accuracy, svm_accuracy, rf_accuracy],
    "Macro F1": [logistic_f1, svm_f1, rf_f1]
}).sort_values("Macro F1", ascending=False).reset_index(drop=True)

display(model_results.style.format({"Accuracy": "{:.4f}", "Macro F1": "{:.4f}"}))

fig = px.bar(
    model_results.melt(id_vars="Model", var_name="Metric", value_name="Score"),
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    title="Model Performance Comparison",
    template="plotly_dark"
)
fig.update_yaxes(range=[0, 1])
fig.show()

## 13. Final Deployable Classifier

Logistic Regression is retained as the deployable classifier because it provides class probabilities required by the web application. The comparison table remains the evidence for discussing whether another model achieved a higher test score.


In [ ]:
final_classifier = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=25000
    )),
    ("model", LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        random_state=42
    ))
])

final_classifier.fit(resume_df["clean_resume"], resume_df["Category"])
joblib.dump(final_classifier, MODEL_DIR / "resume_classifier.joblib")

print("Saved:", MODEL_DIR / "resume_classifier.joblib")

## 14. Job Dataset Acquisition

In [ ]:
try:
    if not job_path.exists():
        response = requests.get(JOB_URL, timeout=60)
        response.raise_for_status()
        job_path.write_bytes(response.content)
    jobs_raw = pd.read_csv(job_path)
    source_used = "Public jobs dataset"
except Exception:
    fallback = [
        ("Data Analyst","Nova Analytics","Remote","Python;SQL;Excel;Power BI;Pandas;Statistics","Analyse business data, build dashboards, write SQL queries, clean datasets and communicate insights."),
        ("Data Scientist","Vertex Labs","Bengaluru","Python;Machine Learning;SQL;Pandas;Scikit-learn;Statistics","Build predictive models, validate experiments and communicate data science findings."),
        ("Machine Learning Engineer","Astra AI","Hyderabad","Python;Machine Learning;Docker;Git;Scikit-learn;APIs","Develop machine learning pipelines, deploy models and monitor prediction services."),
        ("Business Analyst","InsightWorks","Pune","SQL;Excel;Requirements Gathering;Data Analysis;Communication","Translate business requirements into analysis, reporting and process recommendations."),
        ("Python Developer","CodeOrbit","Remote","Python;Git;APIs;SQL;Flask;Django","Develop Python applications, APIs and database-driven backend services."),
        ("Backend Developer","CloudArc","Chennai","Python;SQL;PostgreSQL;APIs;Docker;Git","Develop secure backend services, APIs and database integrations."),
        ("Data Engineer","PipelineX","Bengaluru","Python;SQL;ETL;Spark;Airflow;AWS","Build scalable data pipelines and maintain analytics-ready datasets."),
        ("DevOps Engineer","OpsNova","Remote","Docker;Kubernetes;AWS;Linux;Git;CI/CD","Build automated deployment pipelines and maintain cloud infrastructure."),
        ("Cyber Security Analyst","SecureGrid","Gurugram","Linux;Python;SIEM;Network Security;Incident Response","Monitor security events, investigate incidents and improve defensive controls."),
        ("Database Administrator","DataCore","Noida","SQL;MySQL;PostgreSQL;Database;Linux","Manage relational databases, access controls, backups and query performance."),
        ("Java Developer","ByteForge","Pune","Java;Spring;SQL;Git;APIs","Build Java backend services and maintain enterprise applications."),
        ("Web Developer","PixelStack","Remote","HTML;CSS;JavaScript;React;Git;APIs","Develop responsive web applications and integrate backend APIs."),
        ("Automation Test Engineer","QualityX","Hyderabad","Python;Selenium;Testing;Git;APIs","Design automated tests and improve software quality pipelines."),
        ("Network Engineer","NetWave","Mumbai","Networking;Linux;TCP/IP;Firewalls;Troubleshooting","Configure and troubleshoot enterprise networks and security controls."),
        ("Cloud Engineer","SkyOps","Bengaluru","AWS;Azure;Docker;Linux;Terraform;Networking","Build and operate secure cloud infrastructure and deployment environments."),
        ("AI Engineer","NeuralWorks","Remote","Python;Machine Learning;NLP;Deep Learning;PyTorch;APIs","Develop AI features using machine learning, NLP and production APIs.")
    ]
    jobs_raw = pd.DataFrame(fallback, columns=["Job Title","Company","Location","Skills","Job Description"])
    source_used = "Curated fallback catalogue"

print(source_used)
print("Shape:", jobs_raw.shape)
display(jobs_raw.head())

## 15. Job Data Standardisation

In [ ]:
jobs = jobs_raw.copy()
jobs.columns = [
    re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")
    for col in jobs.columns
]

rename_map = {
    "job_title": "title",
    "jobtitle": "title",
    "title": "title",
    "job_description": "description",
    "jobdescription": "description",
    "description": "description",
    "skills": "skills",
    "company": "company",
    "company_name": "company",
    "location": "location",
    "job_location": "location",
    "certifications": "certifications"
}

jobs = jobs.rename(columns={c: rename_map[c] for c in jobs.columns if c in rename_map})

if "title" not in jobs:
    jobs["title"] = "Job Opportunity"
if "description" not in jobs:
    jobs["description"] = ""
if "skills" not in jobs:
    jobs["skills"] = ""
if "company" not in jobs:
    jobs["company"] = "Not specified"
if "location" not in jobs:
    jobs["location"] = "Not specified"
if "certifications" not in jobs:
    jobs["certifications"] = ""

jobs = jobs[["title", "company", "location", "skills", "description", "certifications"]].copy()

for col in jobs.columns:
    jobs[col] = jobs[col].fillna("").astype(str).str.strip()

jobs = jobs[(jobs["title"] != "") & ((jobs["description"] != "") | (jobs["skills"] != ""))]
jobs = jobs.drop_duplicates(subset=["title", "company", "description"]).reset_index(drop=True)

if len(jobs) > 5000:
    jobs = jobs.sample(5000, random_state=42).reset_index(drop=True)

print("Clean job rows:", len(jobs))
display(jobs.head())

## 16. Skill Vocabulary

In [ ]:
skills = [
    "Python","Java","JavaScript","TypeScript","C++","C#","R","SQL","MySQL","PostgreSQL",
    "MongoDB","SQLite","Oracle","HTML","CSS","React","Angular","Vue","Node.js","Django",
    "Flask","FastAPI","Spring","Git","GitHub","Linux","Docker","Kubernetes","AWS","Azure",
    "GCP","Terraform","Jenkins","CI/CD","REST API","APIs","Pandas","NumPy","Matplotlib",
    "Plotly","Scikit-learn","TensorFlow","Keras","PyTorch","Machine Learning","Deep Learning",
    "NLP","Natural Language Processing","Computer Vision","Statistics","Probability","Regression",
    "Classification","Clustering","Time Series","Forecasting","Data Analysis","Data Analytics",
    "Data Science","Data Visualization","ETL","Data Engineering","Spark","Hadoop","Airflow",
    "Kafka","Power BI","Tableau","Excel","Business Analysis","Business Analyst","Requirements Gathering",
    "Agile","Scrum","Jira","Communication","Problem Solving","Leadership","Project Management",
    "Testing","Selenium","Automation Testing","Manual Testing","PyTest","Cyber Security",
    "Network Security","Networking","SIEM","Incident Response","Firewalls","TCP/IP","DevOps",
    "Cloud Computing","Database","Data Structures","Algorithms","OOP","Object Oriented Programming",
    "Microservices","Software Engineering","Data Modelling","Feature Engineering","Model Deployment",
    "MLOps","Data Cleaning","EDA","Exploratory Data Analysis","GitLab","Redis","Elasticsearch",
    "NoSQL","Bash","Shell Scripting","React Native","Android","Kotlin","PHP","Bootstrap",
    "Figma","UI/UX","CRM","Salesforce","SAP","PowerPoint","Word","Critical Thinking"
]

skills = sorted(set(skills), key=str.lower)
print("Skill vocabulary size:", len(skills))

## 17. Job Skill Extraction

In [ ]:
job_text = (
    jobs["title"] + " " +
    jobs["skills"] + " " +
    jobs["description"] + " " +
    jobs["certifications"]
).apply(clean_text)

detected_job_skills = []

for text in job_text:
    found = []
    padded = " " + text + " "
    for skill in skills:
        pattern = r"(?<![a-z0-9])" + re.escape(skill.lower()) + r"(?![a-z0-9])"
        if re.search(pattern, padded):
            found.append(skill)
    detected_job_skills.append(sorted(set(found), key=str.lower))

jobs["clean_text"] = job_text
jobs["detected_skills"] = ["|".join(x) for x in detected_job_skills]

display(jobs[["title", "skills", "detected_skills"]].head(10))

## 18. Job Exploratory Data Analysis

In [ ]:
role_counts = jobs["title"].value_counts().head(20).reset_index()
role_counts.columns = ["Role", "Count"]

fig = px.bar(
    role_counts.sort_values("Count"),
    x="Count",
    y="Role",
    orientation="h",
    title="Top Job Roles",
    template="plotly_dark"
)
fig.update_layout(height=650)
fig.show()

In [ ]:
skill_series = jobs["detected_skills"].str.split("|").explode().str.strip()
skill_series = skill_series[skill_series.ne("")]
skill_counts = skill_series.value_counts().head(25).reset_index()
skill_counts.columns = ["Skill", "Count"]

fig = px.bar(
    skill_counts.sort_values("Count"),
    x="Count",
    y="Skill",
    orientation="h",
    title="Most Frequent Skills in Job Profiles",
    template="plotly_dark"
)
fig.update_layout(height=700)
fig.show()

## 19. TF-IDF Job Matching Engine

In [ ]:
job_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.98,
    sublinear_tf=True,
    max_features=30000
)

job_matrix = job_vectorizer.fit_transform(jobs["clean_text"])

print("Job matrix shape:", job_matrix.shape)

## 20. Resume Skill Extraction and Recommendation Test

In [ ]:
sample_resume = resume_df.iloc[0]["Resume"]
sample_clean = clean_text(sample_resume)

sample_skills = []
padded_resume = " " + sample_clean + " "

for skill in skills:
    pattern = r"(?<![a-z0-9])" + re.escape(skill.lower()) + r"(?![a-z0-9])"
    if re.search(pattern, padded_resume):
        sample_skills.append(skill)

resume_vector = job_vectorizer.transform([sample_clean])
semantic_scores = cosine_similarity(resume_vector, job_matrix).ravel()

resume_skill_set = {x.lower() for x in sample_skills}
skill_scores = []
matched_skills = []
missing_skills = []

for value in jobs["detected_skills"]:
    current_skills = [x.strip() for x in value.split("|") if x.strip()]
    current_set = {x.lower() for x in current_skills}
    overlap = resume_skill_set & current_set
    skill_scores.append(len(overlap) / max(len(current_set), 1))
    matched_skills.append(sorted(overlap))
    missing_skills.append(sorted(current_set - resume_skill_set))

recommendations = jobs.copy()
recommendations["semantic_score"] = semantic_scores
recommendations["skill_score"] = skill_scores
recommendations["match_score"] = (
    0.72 * recommendations["semantic_score"] +
    0.28 * recommendations["skill_score"]
) * 100
recommendations["matched_skills"] = matched_skills
recommendations["missing_skills"] = missing_skills

display(
    recommendations.sort_values("match_score", ascending=False)[
        ["title", "company", "location", "match_score", "matched_skills", "missing_skills"]
    ].head(10)
)

## 21. Matching Evaluation with Career Category Profiles

In [ ]:
profile_train, profile_test = train_test_split(
    resume_df[["Category", "clean_resume"]],
    test_size=0.20,
    random_state=42,
    stratify=resume_df["Category"]
)

category_profiles = (
    profile_train.groupby("Category")["clean_resume"]
    .apply(lambda values: " ".join(values.tolist()))
    .reset_index()
)

profile_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=30000
)

profile_matrix = profile_vectorizer.fit_transform(category_profiles["clean_resume"])
test_matrix = profile_vectorizer.transform(profile_test["clean_resume"])

profile_similarity = cosine_similarity(test_matrix, profile_matrix)
profile_predictions = category_profiles.iloc[profile_similarity.argmax(axis=1)]["Category"].to_numpy()

matching_accuracy = accuracy_score(profile_test["Category"], profile_predictions)
matching_macro_f1 = f1_score(profile_test["Category"], profile_predictions, average="macro")

print("Category retrieval accuracy:", round(matching_accuracy, 4))
print("Category retrieval macro F1:", round(matching_macro_f1, 4))
print(classification_report(profile_test["Category"], profile_predictions, zero_division=0))

## 22. Matching Evaluation Visual

In [ ]:
matching_cm = confusion_matrix(
    profile_test["Category"],
    profile_predictions,
    labels=category_profiles["Category"]
)

fig = px.imshow(
    matching_cm,
    x=category_profiles["Category"],
    y=category_profiles["Category"],
    labels=dict(x="Predicted Profile", y="Actual Category", color="Count"),
    title="Resume-to-Category Matching Confusion Matrix",
    template="plotly_dark",
    aspect="auto"
)
fig.update_layout(height=850)
fig.show()

## 23. Export Final Models and Processed Data

In [ ]:
jobs.to_csv(DATA_DIR / "jobs_clean.csv", index=False)
(DATA_DIR / "skills.json").write_text(json.dumps(skills, indent=2), encoding="utf-8")

joblib.dump(job_vectorizer, MODEL_DIR / "job_vectorizer.joblib")
joblib.dump(job_matrix, MODEL_DIR / "job_matrix.joblib")

print("Saved jobs:", DATA_DIR / "jobs_clean.csv")
print("Saved skills:", DATA_DIR / "skills.json")
print("Saved vectorizer:", MODEL_DIR / "job_vectorizer.joblib")
print("Saved job matrix:", MODEL_DIR / "job_matrix.joblib")

## 24. SQLite Database

In [ ]:
db_path = ROOT / "career_lens.db"

with sqlite3.connect(db_path) as conn:
    conn.execute("""
    CREATE TABLE IF NOT EXISTS users(
        user_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        salt TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    conn.execute("""
    CREATE TABLE IF NOT EXISTS analyses(
        analysis_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        predicted_role TEXT,
        detected_skills TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    conn.execute("""
    CREATE TABLE IF NOT EXISTS applications(
        application_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        job_title TEXT NOT NULL,
        company TEXT,
        status TEXT NOT NULL,
        notes TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    conn.execute("""
    CREATE TABLE IF NOT EXISTS feedback(
        feedback_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        rating INTEGER,
        comments TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    conn.commit()

with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn
    )

display(tables)

## 25. Generate Streamlit Application

In [ ]:
streamlit_code = 'import io\nimport json\nimport hashlib\nimport os\nimport re\nimport sqlite3\nfrom pathlib import Path\n\nimport fitz\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport plotly.express as px\nimport streamlit as st\nfrom docx import Document\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nst.set_page_config(page_title="CareerLens AI", page_icon="✨", layout="wide", initial_sidebar_state="expanded")\n\nROOT = Path(__file__).resolve().parent\nDATA_DIR = ROOT / "data"\nMODEL_DIR = ROOT / "models"\nDB_PATH = ROOT / "career_lens.db"\n\nst.markdown("""\n<style>\n@import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap\');\nhtml, body, [class*="css"] {font-family: \'Inter\', sans-serif;}\n.stApp {\n    background:\n        radial-gradient(circle at 15% 5%, rgba(124,58,237,.24), transparent 28%),\n        radial-gradient(circle at 88% 12%, rgba(14,165,233,.18), transparent 26%),\n        linear-gradient(135deg,#070913 0%,#0B1020 45%,#090B14 100%);\n}\n.block-container {padding-top: 1.5rem; padding-bottom: 3rem;}\n.hero {\n    padding: 2.2rem 2.3rem;\n    border-radius: 28px;\n    background: linear-gradient(120deg,rgba(124,58,237,.22),rgba(14,165,233,.14));\n    border: 1px solid rgba(255,255,255,.10);\n    box-shadow: 0 24px 70px rgba(0,0,0,.28);\n    backdrop-filter: blur(14px);\n}\n.hero h1 {\n    font-size: 3.1rem;\n    line-height: 1.05;\n    margin: 0 0 .7rem 0;\n    background: linear-gradient(90deg,#FFFFFF,#C4B5FD,#7DD3FC);\n    -webkit-background-clip: text;\n    -webkit-text-fill-color: transparent;\n}\n.hero p {font-size:1.05rem;color:#CBD5E1;max-width:850px;margin:0;}\n.badge {\n    display:inline-block;\n    padding:.38rem .7rem;\n    border-radius:999px;\n    margin:.18rem .2rem .18rem 0;\n    background:rgba(124,58,237,.16);\n    border:1px solid rgba(196,181,253,.24);\n    color:#DDD6FE;\n    font-size:.82rem;\n}\n.glass {\n    padding:1.15rem 1.2rem;\n    border-radius:20px;\n    background:rgba(17,24,39,.72);\n    border:1px solid rgba(255,255,255,.08);\n    box-shadow:0 12px 38px rgba(0,0,0,.20);\n}\n.metric-card {\n    padding:1.1rem;\n    border-radius:18px;\n    background:linear-gradient(145deg,rgba(30,41,59,.90),rgba(15,23,42,.78));\n    border:1px solid rgba(255,255,255,.08);\n    min-height:115px;\n}\n.metric-title {color:#94A3B8;font-size:.83rem;font-weight:600;}\n.metric-value {font-size:1.9rem;font-weight:800;color:#F8FAFC;margin-top:.25rem;}\n.metric-sub {color:#A5B4FC;font-size:.78rem;margin-top:.2rem;}\n.job-card {\n    padding:1.05rem 1.15rem;\n    margin:.65rem 0;\n    border-radius:18px;\n    background:rgba(15,23,42,.82);\n    border:1px solid rgba(148,163,184,.13);\n}\n.score {\n    font-size:1.65rem;\n    font-weight:800;\n    background:linear-gradient(90deg,#C4B5FD,#67E8F9);\n    -webkit-background-clip:text;\n    -webkit-text-fill-color:transparent;\n}\n.small-muted {color:#94A3B8;font-size:.84rem;}\n[data-testid="stSidebar"] {\n    background:linear-gradient(180deg,#0B1020,#090B14);\n    border-right:1px solid rgba(255,255,255,.07);\n}\ndiv.stButton > button {\n    border-radius:12px;\n    border:1px solid rgba(196,181,253,.32);\n    background:linear-gradient(90deg,#6D28D9,#2563EB);\n    color:white;\n    font-weight:700;\n}\ndiv.stButton > button:hover {border-color:#A78BFA;}\n[data-testid="stFileUploader"] {\n    border:1px dashed rgba(167,139,250,.45);\n    border-radius:18px;\n    padding:.6rem;\n}\n</style>\n""", unsafe_allow_html=True)\n\n@st.cache_resource\ndef load_assets():\n    classifier = joblib.load(MODEL_DIR / "resume_classifier.joblib")\n    vectorizer = joblib.load(MODEL_DIR / "job_vectorizer.joblib")\n    job_matrix = joblib.load(MODEL_DIR / "job_matrix.joblib")\n    jobs = pd.read_csv(DATA_DIR / "jobs_clean.csv")\n    skills = json.loads((DATA_DIR / "skills.json").read_text(encoding="utf-8"))\n    return classifier, vectorizer, job_matrix, jobs, skills\n\ndef db_connect():\n    return sqlite3.connect(DB_PATH)\n\ndef init_db():\n    with db_connect() as conn:\n        conn.execute("""\n        CREATE TABLE IF NOT EXISTS users(\n            user_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            name TEXT NOT NULL,\n            email TEXT UNIQUE NOT NULL,\n            password_hash TEXT NOT NULL,\n            salt TEXT NOT NULL,\n            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n        )\n        """)\n        conn.execute("""\n        CREATE TABLE IF NOT EXISTS analyses(\n            analysis_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            user_id INTEGER,\n            predicted_role TEXT,\n            detected_skills TEXT,\n            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n        )\n        """)\n        conn.execute("""\n        CREATE TABLE IF NOT EXISTS applications(\n            application_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            user_id INTEGER,\n            job_title TEXT NOT NULL,\n            company TEXT,\n            status TEXT NOT NULL,\n            notes TEXT,\n            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n        )\n        """)\n        conn.commit()\n\ndef password_hash(password, salt):\n    return hashlib.pbkdf2_hmac("sha256", password.encode(), bytes.fromhex(salt), 120000).hex()\n\ndef register_user(name, email, password):\n    salt = os.urandom(16).hex()\n    secure_hash = password_hash(password, salt)\n    try:\n        with db_connect() as conn:\n            conn.execute(\n                "INSERT INTO users(name,email,password_hash,salt) VALUES(?,?,?,?)",\n                (name.strip(), email.strip().lower(), secure_hash, salt)\n            )\n            conn.commit()\n        return True, "Account created successfully."\n    except sqlite3.IntegrityError:\n        return False, "An account with this email already exists."\n\ndef login_user(email, password):\n    with db_connect() as conn:\n        row = conn.execute(\n            "SELECT user_id,name,email,password_hash,salt FROM users WHERE email=?",\n            (email.strip().lower(),)\n        ).fetchone()\n    if row and password_hash(password, row[4]) == row[3]:\n        return {"user_id": row[0], "name": row[1], "email": row[2]}\n    return None\n\ndef extract_text(uploaded_file):\n    content = uploaded_file.getvalue()\n    suffix = Path(uploaded_file.name).suffix.lower()\n    if suffix == ".pdf":\n        doc = fitz.open(stream=content, filetype="pdf")\n        return "\\n".join(page.get_text("text") for page in doc)\n    if suffix == ".docx":\n        doc = Document(io.BytesIO(content))\n        return "\\n".join(p.text for p in doc.paragraphs)\n    if suffix == ".txt":\n        return content.decode("utf-8", errors="ignore")\n    return ""\n\ndef clean_text(text):\n    text = str(text).lower()\n    text = re.sub(r"http\\S+|www\\S+|https\\S+", " ", text)\n    text = re.sub(r"\\S+@\\S+", " ", text)\n    text = re.sub(r"[^a-z0-9+#.\\s-]", " ", text)\n    text = re.sub(r"\\s+", " ", text).strip()\n    return text\n\ndef detect_skills(text, skills):\n    text_clean = " " + clean_text(text) + " "\n    found = []\n    for skill in skills:\n        pattern = r"(?<![a-z0-9])" + re.escape(skill.lower()) + r"(?![a-z0-9])"\n        if re.search(pattern, text_clean):\n            found.append(skill)\n    return sorted(set(found), key=str.lower)\n\ndef job_recommendations(resume_text, resume_skills, jobs, vectorizer, matrix):\n    resume_vector = vectorizer.transform([clean_text(resume_text)])\n    semantic = cosine_similarity(resume_vector, matrix).ravel()\n    resume_skill_set = {s.lower() for s in resume_skills}\n    skill_scores = []\n    matched = []\n    missing = []\n    for value in jobs["detected_skills"].fillna(""):\n        job_skills = [x.strip() for x in value.split("|") if x.strip()]\n        job_set = {x.lower() for x in job_skills}\n        overlap = resume_skill_set & job_set\n        score = len(overlap) / max(len(job_set), 1)\n        skill_scores.append(score)\n        matched.append(sorted(overlap))\n        missing.append(sorted(job_set - resume_skill_set))\n    result = jobs.copy()\n    result["semantic_score"] = semantic\n    result["skill_score"] = np.array(skill_scores)\n    result["match_score"] = (0.72 * result["semantic_score"] + 0.28 * result["skill_score"]) * 100\n    result["matched_skills"] = matched\n    result["missing_skills"] = missing\n    return result.sort_values("match_score", ascending=False).reset_index(drop=True)\n\ndef save_analysis(user_id, role, skills):\n    with db_connect() as conn:\n        conn.execute(\n            "INSERT INTO analyses(user_id,predicted_role,detected_skills) VALUES(?,?,?)",\n            (user_id, role, " | ".join(skills))\n        )\n        conn.commit()\n\ninit_db()\n\ntry:\n    classifier, vectorizer, job_matrix, jobs, skills = load_assets()\nexcept Exception:\n    st.error("Project assets are missing. Run the complete notebook first, then restart this app.")\n    st.stop()\n\nif "user" not in st.session_state:\n    st.session_state.user = None\nif "guest" not in st.session_state:\n    st.session_state.guest = False\n\nwith st.sidebar:\n    st.markdown("## ✨ CareerLens AI")\n    st.caption("Your intelligent career navigator")\n    if st.session_state.user:\n        st.success(f"Signed in as {st.session_state.user[\'name\']}")\n    elif st.session_state.guest:\n        st.info("Guest mode")\n    page = st.radio(\n        "Navigate",\n        ["Home", "Analyse Resume", "Job Explorer", "Applications", "Insights", "Methodology"],\n        label_visibility="collapsed"\n    )\n    st.divider()\n    if st.session_state.user or st.session_state.guest:\n        if st.button("Sign out", use_container_width=True):\n            st.session_state.user = None\n            st.session_state.guest = False\n            st.rerun()\n\nif not st.session_state.user and not st.session_state.guest:\n    st.markdown("""\n    <div class="hero">\n        <span class="badge">NLP</span><span class="badge">Machine Learning</span><span class="badge">Skill Intelligence</span>\n        <h1>Turn your resume into a career strategy.</h1>\n        <p>CareerLens AI predicts suitable career categories, matches your resume with jobs, identifies missing skills and turns the result into a clear action plan.</p>\n    </div>\n    """, unsafe_allow_html=True)\n    st.write("")\n    login_tab, register_tab, guest_tab = st.tabs(["Sign in", "Create account", "Explore as guest"])\n    with login_tab:\n        with st.form("login_form"):\n            email = st.text_input("Email")\n            password = st.text_input("Password", type="password")\n            submitted = st.form_submit_button("Sign in", use_container_width=True)\n        if submitted:\n            user = login_user(email, password)\n            if user:\n                st.session_state.user = user\n                st.rerun()\n            st.error("Invalid email or password.")\n    with register_tab:\n        with st.form("register_form"):\n            name = st.text_input("Full name")\n            email = st.text_input("Email", key="register_email")\n            password = st.text_input("Password", type="password", key="register_password")\n            submitted = st.form_submit_button("Create account", use_container_width=True)\n        if submitted:\n            if len(name.strip()) < 2 or "@" not in email or len(password) < 6:\n                st.error("Enter a valid name, email and password with at least 6 characters.")\n            else:\n                ok, message = register_user(name, email, password)\n                if ok:\n                    st.success(message)\n                else:\n                    st.error(message)\n    with guest_tab:\n        st.write("Open the full application without creating an account.")\n        if st.button("Continue as guest", use_container_width=True):\n            st.session_state.guest = True\n            st.rerun()\n    st.stop()\n\nif page == "Home":\n    st.markdown("""\n    <div class="hero">\n        <span class="badge">AI Resume Intelligence</span><span class="badge">Explainable Matching</span><span class="badge">Python + ML</span>\n        <h1>Find roles that fit. See exactly what is missing.</h1>\n        <p>Upload a resume and receive role prediction, job-match scores, matched skills, missing skills and a focused improvement path in one place.</p>\n    </div>\n    """, unsafe_allow_html=True)\n    st.write("")\n    c1, c2, c3, c4 = st.columns(4)\n    values = [\n        ("Job records", f"{len(jobs):,}", "searchable recommendation pool"),\n        ("Job categories", f"{jobs[\'title\'].nunique():,}", "different role labels"),\n        ("Skill vocabulary", f"{len(skills):,}", "technical and business skills"),\n        ("Matching engine", "Hybrid", "TF-IDF + skill coverage")\n    ]\n    for col, item in zip([c1, c2, c3, c4], values):\n        with col:\n            st.markdown(\n                f\'<div class="metric-card"><div class="metric-title">{item[0]}</div><div class="metric-value">{item[1]}</div><div class="metric-sub">{item[2]}</div></div>\',\n                unsafe_allow_html=True\n            )\n    st.write("")\n    left, right = st.columns([1.1, .9])\n    with left:\n        st.markdown("### What CareerLens does")\n        st.markdown("""\n        <div class="glass">\n        <b>1.</b> Reads PDF, DOCX or TXT resumes<br><br>\n        <b>2.</b> Predicts the most suitable resume category using machine learning<br><br>\n        <b>3.</b> Detects relevant skills using NLP-style text matching<br><br>\n        <b>4.</b> Ranks jobs using TF-IDF similarity and skill coverage<br><br>\n        <b>5.</b> Explains the recommendation through matched and missing skills\n        </div>\n        """, unsafe_allow_html=True)\n    with right:\n        role_counts = jobs["title"].value_counts().head(10).reset_index()\n        role_counts.columns = ["Role", "Jobs"]\n        fig = px.bar(role_counts, x="Jobs", y="Role", orientation="h", template="plotly_dark", title="Largest Job Groups")\n        fig.update_layout(height=430, margin=dict(l=10, r=10, t=50, b=10), yaxis={"categoryorder":"total ascending"})\n        st.plotly_chart(fig, use_container_width=True)\n\nelif page == "Analyse Resume":\n    st.markdown("## Resume Intelligence")\n    st.caption("Upload one resume. CareerLens will analyse it against the trained classifier and job corpus.")\n    uploaded = st.file_uploader("Upload resume", type=["pdf", "docx", "txt"])\n    if uploaded:\n        resume_text = extract_text(uploaded)\n        if len(resume_text.strip()) < 80:\n            st.error("Very little text could be extracted. Try another PDF, DOCX or TXT file.")\n        else:\n            cleaned = clean_text(resume_text)\n            predicted_role = classifier.predict([cleaned])[0]\n            probabilities = classifier.predict_proba([cleaned])[0]\n            classes = classifier.classes_\n            top_idx = np.argsort(probabilities)[::-1][:5]\n            top_roles = pd.DataFrame({"Role": classes[top_idx], "Confidence": probabilities[top_idx] * 100})\n            resume_skills = detect_skills(resume_text, skills)\n            recommendations = job_recommendations(resume_text, resume_skills, jobs, vectorizer, job_matrix)\n            best = recommendations.iloc[0]\n            c1, c2, c3, c4 = st.columns(4)\n            with c1:\n                st.markdown(f\'<div class="metric-card"><div class="metric-title">Predicted role</div><div class="metric-value" style="font-size:1.25rem">{predicted_role}</div><div class="metric-sub">highest classifier probability</div></div>\', unsafe_allow_html=True)\n            with c2:\n                st.markdown(f\'<div class="metric-card"><div class="metric-title">Detected skills</div><div class="metric-value">{len(resume_skills)}</div><div class="metric-sub">recognised from vocabulary</div></div>\', unsafe_allow_html=True)\n            with c3:\n                st.markdown(f\'<div class="metric-card"><div class="metric-title">Best job match</div><div class="metric-value">{best["match_score"]:.1f}%</div><div class="metric-sub">{best["title"]}</div></div>\', unsafe_allow_html=True)\n            with c4:\n                st.markdown(f\'<div class="metric-card"><div class="metric-title">Priority gaps</div><div class="metric-value">{len(best["missing_skills"])}</div><div class="metric-sub">skills missing from top match</div></div>\', unsafe_allow_html=True)\n            st.write("")\n            tab1, tab2, tab3 = st.tabs(["Top Matches", "Skill Gap", "Role Prediction"])\n            with tab1:\n                for _, row in recommendations.head(7).iterrows():\n                    matched_text = ", ".join(row["matched_skills"][:8]) if row["matched_skills"] else "No exact skill overlap detected"\n                    st.markdown(\n                        f\'<div class="job-card"><div class="score">{row["match_score"]:.1f}% match</div><h3 style="margin:.15rem 0">{row["title"]}</h3><div class="small-muted">{row["company"]} · {row["location"]}</div><div style="margin-top:.7rem"><b>Matched skills:</b> {matched_text}</div></div>\',\n                        unsafe_allow_html=True\n                    )\n            with tab2:\n                top = recommendations.iloc[0]\n                l, r = st.columns(2)\n                with l:\n                    st.success("Matched skills")\n                    if top["matched_skills"]:\n                        st.markdown(" ".join([f\'<span class="badge">{x}</span>\' for x in top["matched_skills"]]), unsafe_allow_html=True)\n                    else:\n                        st.write("No exact skill overlap found.")\n                with r:\n                    st.warning("Skills to strengthen")\n                    if top["missing_skills"]:\n                        st.markdown(" ".join([f\'<span class="badge">{x}</span>\' for x in top["missing_skills"][:18]]), unsafe_allow_html=True)\n                    else:\n                        st.write("No skill gaps were detected for the top recommendation.")\n                gap_data = pd.DataFrame({"Area": ["Matched", "Missing"], "Count": [len(top["matched_skills"]), len(top["missing_skills"])]})\n                fig = px.bar(gap_data, x="Area", y="Count", template="plotly_dark", title=f"Skill Coverage for {top[\'title\']}")\n                st.plotly_chart(fig, use_container_width=True)\n            with tab3:\n                fig = px.bar(top_roles.sort_values("Confidence"), x="Confidence", y="Role", orientation="h", template="plotly_dark", title="Top Predicted Career Categories")\n                st.plotly_chart(fig, use_container_width=True)\n                st.dataframe(top_roles.style.format({"Confidence":"{:.2f}%"}), use_container_width=True, hide_index=True)\n            if st.session_state.user:\n                save_analysis(st.session_state.user["user_id"], predicted_role, resume_skills)\n\nelif page == "Job Explorer":\n    st.markdown("## Job Explorer")\n    c1, c2 = st.columns([1.4, .6])\n    with c1:\n        query = st.text_input("Search job title, skill or description", placeholder="Python, Data Analyst, SQL...")\n    with c2:\n        top_n = st.selectbox("Rows", [20, 50, 100], index=0)\n    shown = jobs.copy()\n    if query.strip():\n        q = query.lower().strip()\n        mask = (\n            shown["title"].fillna("").str.lower().str.contains(q, regex=False) |\n            shown["description"].fillna("").str.lower().str.contains(q, regex=False) |\n            shown["detected_skills"].fillna("").str.lower().str.contains(q, regex=False)\n        )\n        shown = shown[mask]\n    st.metric("Jobs found", f"{len(shown):,}")\n    st.dataframe(shown[["title", "company", "location", "detected_skills"]].head(top_n), use_container_width=True, hide_index=True)\n\nelif page == "Applications":\n    st.markdown("## Application Tracker")\n    if not st.session_state.user:\n        st.info("Create an account and sign in to keep application records linked to your profile.")\n    else:\n        user_id = st.session_state.user["user_id"]\n        with st.form("application_form"):\n            title = st.text_input("Job title")\n            company = st.text_input("Company")\n            status = st.selectbox("Status", ["Saved", "Applied", "Assessment", "Interview", "Offer", "Rejected"])\n            notes = st.text_area("Notes")\n            submitted = st.form_submit_button("Add application", use_container_width=True)\n        if submitted and title.strip():\n            with db_connect() as conn:\n                conn.execute(\n                    "INSERT INTO applications(user_id,job_title,company,status,notes) VALUES(?,?,?,?,?)",\n                    (user_id, title.strip(), company.strip(), status, notes.strip())\n                )\n                conn.commit()\n            st.success("Application saved.")\n        with db_connect() as conn:\n            apps = pd.read_sql_query(\n                "SELECT application_id,job_title,company,status,notes,created_at FROM applications WHERE user_id=? ORDER BY application_id DESC",\n                conn,\n                params=(user_id,)\n            )\n        if len(apps):\n            status_counts = apps["status"].value_counts().reset_index()\n            status_counts.columns = ["Status", "Count"]\n            fig = px.pie(status_counts, names="Status", values="Count", hole=.62, template="plotly_dark", title="Application Pipeline")\n            st.plotly_chart(fig, use_container_width=True)\n            st.dataframe(apps, use_container_width=True, hide_index=True)\n        else:\n            st.info("No applications have been added yet.")\n\nelif page == "Insights":\n    st.markdown("## Job Market Insights")\n    col1, col2 = st.columns(2)\n    with col1:\n        role_counts = jobs["title"].value_counts().head(15).reset_index()\n        role_counts.columns = ["Role", "Count"]\n        fig = px.bar(role_counts.sort_values("Count"), x="Count", y="Role", orientation="h", template="plotly_dark", title="Most Frequent Job Roles")\n        st.plotly_chart(fig, use_container_width=True)\n    with col2:\n        exploded = jobs["detected_skills"].fillna("").str.split("|").explode().str.strip()\n        exploded = exploded[exploded.ne("")]\n        skill_counts = exploded.value_counts().head(15).reset_index()\n        skill_counts.columns = ["Skill", "Count"]\n        fig = px.bar(skill_counts.sort_values("Count"), x="Count", y="Skill", orientation="h", template="plotly_dark", title="Most Requested Skills")\n        st.plotly_chart(fig, use_container_width=True)\n    st.markdown("### Dataset overview")\n    c1, c2, c3 = st.columns(3)\n    c1.metric("Jobs", f"{len(jobs):,}")\n    c2.metric("Unique titles", f"{jobs[\'title\'].nunique():,}")\n    c3.metric("Unique companies", f"{jobs[\'company\'].nunique():,}")\n\nelif page == "Methodology":\n    st.markdown("## Methodology")\n    st.markdown("""\n    <div class="glass">\n    <h3>1. Resume Parsing</h3>\n    PDF files are read with PyMuPDF, DOCX files with python-docx and TXT files directly.<br><br>\n    <h3>2. NLP Pre-processing</h3>\n    Resume and job text is normalised, cleaned and represented using TF-IDF n-gram features.<br><br>\n    <h3>3. Machine Learning Classification</h3>\n    The classifier predicts the most suitable resume category from labelled training resumes.<br><br>\n    <h3>4. Hybrid Matching</h3>\n    Each resume receives a recommendation score combining TF-IDF cosine similarity and explicit skill coverage.<br><br>\n    <h3>5. Explainable Skill Gap</h3>\n    The recommendation shows both matched skills and skills present in the job profile but not detected in the resume.\n    </div>\n    """, unsafe_allow_html=True)\n'

(ROOT / "streamlit_app.py").write_text(streamlit_code, encoding="utf-8")

print("Created:", ROOT / "streamlit_app.py")

## 26. Generate Deployment Files

In [ ]:
requirements_text = 'streamlit>=1.50,<2\npandas>=2.2,<3\nnumpy>=2.0,<3\nscikit-learn>=1.6,<2\nplotly>=6,<7\nmatplotlib>=3.9,<4\nwordcloud>=1.9,<2\njoblib>=1.4,<2\nPyMuPDF>=1.26,<2\npython-docx>=1.2,<2\nrequests>=2.32,<3\n'
config_text = '[theme]\nbase="dark"\nprimaryColor="#7C3AED"\nbackgroundColor="#090B14"\nsecondaryBackgroundColor="#121827"\ntextColor="#F8FAFC"\nfont="sans serif"\n\n[server]\nheadless=true\n'
gitignore_text = '__pycache__/\n.ipynb_checkpoints/\n.venv/\nvenv/\n.env\n*.pyc\n*.pyo\n.DS_Store\ncareer_lens.db\n'

(ROOT / "requirements.txt").write_text(requirements_text, encoding="utf-8")
(STREAMLIT_DIR / "config.toml").write_text(config_text, encoding="utf-8")
(ROOT / ".gitignore").write_text(gitignore_text, encoding="utf-8")

print("Created requirements.txt")
print("Created .streamlit/config.toml")
print("Created .gitignore")

## 27. Final Project Verification

In [ ]:
required_files = [
    ROOT / "streamlit_app.py",
    ROOT / "requirements.txt",
    ROOT / ".streamlit" / "config.toml",
    ROOT / "models" / "resume_classifier.joblib",
    ROOT / "models" / "job_vectorizer.joblib",
    ROOT / "models" / "job_matrix.joblib",
    ROOT / "data" / "jobs_clean.csv",
    ROOT / "data" / "skills.json",
    ROOT / "career_lens.db"
]

verification = pd.DataFrame({
    "File": [str(path.relative_to(ROOT)) for path in required_files],
    "Exists": [path.exists() for path in required_files]
})

display(verification)

if verification["Exists"].all():
    print("CareerLens AI practical pipeline is complete.")
else:
    print("Run all notebook cells again from the beginning.")

## 28. Run the Web Application

Open the VS Code terminal in the project folder and run:

```bash
pip install -r requirements.txt
streamlit run streamlit_app.py
```

The local application normally opens at:

```text
http://localhost:8501
```

The notebook is complete when the final verification table shows `True` for every required file.
